In [ ]:
!pip install playwright nest-asyncio pandas
!playwright install chromium

In [ ]:
import asyncio
import nest_asyncio
import pandas as pd
import random
import time
import os
import re
from datetime import datetime, timedelta
from playwright.async_api import async_playwright

nest_asyncio.apply()

# ============================================================
# CONFIG
# ============================================================
INPUT_FILE = "./tr1.csv"
TEMP_OUTPUT_FILE = "hotel_prices_temp.csv"
OUTPUT_PREFIX = "hotel_prices_"

HEADLESS = True
DEBUG_SCREENSHOTS = False
DEBUG_DIR = "./debug_screenshots"

NUM_WORKERS = 4             # 4 hotels song song trong 1 batch
WEEKS_PER_HOTEL = 6         # 3 weeks song song per hotel (tăng từ 2)
DAYS_PER_WEEK = 3           # Số ngày thử trong mỗi tuần (fallback từng ngày)
SOLD_OUT_EARLY_EXIT = 3     # Dừng sớm nếu N ngày liên tiếp SOLD OUT
RETRIES_PER_DAY = 1         # Retry cho mỗi ngày
PAGE_TIMEOUT = 30000
BATCH_SIZE = 8              # 5 hotels/batch — restart browser mỗi batch
MAX_RETRY_ROUNDS = 1
TARGET_NA_RATE = 0.10

AUTO_RETRY_NA_SOLDOUT = True  # Tự động crawl lại NA & SOLD OUT sau round 1

# ── Anti-detection ──
BATCH_COOLDOWN = (10, 20)   # Nghỉ 15-30s giữa các batch (đủ reset rate limiter)
RESTART_BROWSER = True      # Restart browser mỗi batch (xóa sạch fingerprint)

# ── Proxy rotation (optional) ──
# Để trống [] = không dùng proxy (dùng IP thật)
# Thêm proxy vào list để rotate mỗi batch
# Format: "http://host:port" hoặc "socks5://host:port"
# Ví dụ Tor: ["socks5://127.0.0.1:9050"]
PROXY_LIST = []

DELAY_RANGE = (1.5, 3.0)
HOTEL_DELAY = (1, 2)
RETRY_COOL_DOWN = (5, 10)
RETRY_PAGE_TIMEOUT = [30000, 45000]

USER_AGENTS = [
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.2 Safari/605.1.15",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:122.0) Gecko/20100101 Firefox/122.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 14_3) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.3 Safari/605.1.15",
]

STEALTH_SCRIPT = """
Object.defineProperty(navigator, 'webdriver', {get: () => undefined});
Object.defineProperty(navigator, 'plugins', {get: () => [1, 2, 3, 4, 5]});
Object.defineProperty(navigator, 'languages', {get: () => ['en-US', 'en']});
window.chrome = { runtime: {} };
const originalQuery = window.navigator.permissions.query;
window.navigator.permissions.query = (parameters) => (
    parameters.name === 'notifications'
    ? Promise.resolve({state: Notification.permission})
    : originalQuery(parameters)
);
"""

EXTRACT_PRICES_JS = """(targetRoom) => {
    const results = [];
    const masterRooms = document.querySelectorAll("[data-selenium='MasterRoom']");
    const targetLower = targetRoom.toLowerCase().trim();

    function findPriceInText(text) {
        const m1 = text.match(/₫\\s*[\\d,.\\ ]+/);
        if (m1) return m1[0].trim();
        const m2 = text.match(/[\\d,.]+\\s*₫/);
        if (m2) return m2[0].trim();
        const m3 = text.match(/VND\\s*[\\d,.\\ ]+|[\\d,.]+\\s*VND/i);
        if (m3) return m3[0].trim();
        return null;
    }

    const bodyText = document.body.innerText || '';
    const hotelSoldOut = /sold\\s*out[!.]?\\s*(our last room|all rooms)/i.test(bodyText);
    const noAvailability = /no\\s*(rooms?)?\\s*avail/i.test(bodyText) && masterRooms.length === 0;

    if ((hotelSoldOut || noAvailability) && masterRooms.length === 0) {
        return {found: false, soldOut: true, soldOutType: 'hotel', allRooms: 0, pageTitle: document.title, bodySnippet: bodyText.substring(0, 300)};
    }

    masterRooms.forEach(room => {
        const nameEl = room.querySelector("[data-selenium='masterroom-title-name']");
        const name = nameEl ? nameEl.textContent.trim() : '';
        const roomText = room.innerText || '';

        let soldOutPrice = null;
        const soldOutMatch = roomText.match(/sold\\s*out\\s*at\\s*([₫đ]\\s*[\\d,.\\ ]+|[\\d,.]+\\s*[₫đ])/i);
        if (soldOutMatch) soldOutPrice = soldOutMatch[1].trim();
        if (!soldOutPrice) {
            let parent = room.parentElement;
            for (let i = 0; i < 3 && parent; i++) {
                const parentMatch = (parent.innerText || '').match(/sold\\s*out\\s*at\\s*([₫đ]\\s*[\\d,.\\ ]+|[\\d,.]+\\s*[₫đ])/i);
                if (parentMatch) { soldOutPrice = parentMatch[1].trim(); break; }
                parent = parent.parentElement;
            }
        }

        let prices = [];
        room.querySelectorAll("[data-selenium='PriceDisplay']").forEach(p => {
            const text = p.textContent.trim();
            if (text) prices.push(text);
        });
        if (prices.length === 0) {
            let parent = room.parentElement;
            for (let i = 0; i < 3 && parent; i++) {
                parent.querySelectorAll("[data-selenium='PriceDisplay']").forEach(p => {
                    const text = p.textContent.trim();
                    if (text) prices.push(text);
                });
                if (prices.length > 0) break;
                parent = parent.parentElement;
            }
        }
        if (prices.length === 0) {
            const walker = document.createTreeWalker(room, NodeFilter.SHOW_TEXT);
            while (walker.nextNode()) {
                const price = findPriceInText(walker.currentNode.textContent.trim());
                if (price) prices.push(price);
            }
        }

        results.push({
            name, nameLower: name.toLowerCase().trim(),
            prices: prices.slice(0, 5),
            matched: name.toLowerCase().trim() === targetLower,
            soldOutPrice
        });
    });

    const target = results.find(r => r.matched);
    if (target) {
        if (target.prices.length > 0) return {found: true, price: target.prices[0], room: target.name, allRooms: results.length};
        if (target.soldOutPrice) return {found: false, soldOut: true, soldOutType: 'room', soldOutPrice: target.soldOutPrice, room: target.name, allRooms: results.length};
    }
    const partial = results.find(r => r.nameLower.includes(targetLower) || targetLower.includes(r.nameLower));
    if (partial) {
        if (partial.prices.length > 0) return {found: true, price: partial.prices[0], room: partial.name, allRooms: results.length, partial: true};
        if (partial.soldOutPrice) return {found: false, soldOut: true, soldOutType: 'room', soldOutPrice: partial.soldOutPrice, room: partial.name, allRooms: results.length, partial: true};
    }
    const allSoldOut = results.length > 0 && results.every(r => r.soldOutPrice && r.prices.length === 0);
    if (allSoldOut) {
        const rel = target || partial || results[0];
        return {found: false, soldOut: true, soldOutType: 'all_rooms', soldOutPrice: rel.soldOutPrice, allRooms: results.length};
    }
    return {found: false, soldOut: false, allRooms: results.length, roomNames: results.map(r => r.name), pageTitle: document.title, bodySnippet: (document.body.innerText || '').substring(0, 300)};
}"""

# ============================================================
# HELPERS
# ============================================================
def read_hotels_from_csv(file_path):
    try:
        df = pd.read_csv(file_path)
        required_cols = ['hotel_name', 'hotel_url', 'room_type']
        if not all(col in df.columns for col in required_cols):
            df.columns = ['hotel_name', 'hotel_url', 'room_type'] + list(df.columns[3:])
        df = df[df['hotel_url'].notna() & (df['hotel_url'] != '')]
        print(f"✅ {len(df)} hotels từ {file_path}", flush=True)
        return df[['hotel_name', 'hotel_url', 'room_type']]
    except Exception as e:
        print(f"❌ Lỗi đọc CSV: {e}", flush=True)
        return pd.DataFrame(columns=['hotel_name', 'hotel_url', 'room_type'])

def save_backup_csv(all_week_prices, filename):
    try:
        rows = []
        for (hotel, room), prices in all_week_prices.items():
            row = {"hotel_name": hotel, "room_type": room}
            for i in range(1, 7):
                row[f"price_w{i}"] = prices.get(f"Price W{i}", "NA")
            rows.append(row)
        pd.DataFrame(rows).to_csv(filename, index=False)
    except Exception as e:
        print(f"❌ Error saving: {e}", flush=True)

def update_url_checkin(url, checkin_date):
    new_date = checkin_date.strftime("%Y-%m-%d")
    if 'checkin=' in url.lower():
        return re.sub(r'checkin=[\d-]+', f'checkin={new_date}', url, flags=re.IGNORECASE)
    return f"{url}{'&' if '?' in url else '?'}checkin={new_date}"

def calc_na_stats(d):
    return sum(1 for i in range(1, 7) if d.get(f"Price W{i}", "NA") == "NA"), 6

def calc_batch_na_rate(awp, keys):
    tc, nc = 0, 0
    for k in keys:
        if k in awp:
            n, t = calc_na_stats(awp[k])
            nc += n; tc += t
    return nc / max(tc, 1), nc, tc

def find_retry_weeks(awp, keys):
    items = []
    for k in keys:
        if k not in awp: continue
        p = awp[k]
        has_real = any(v != "NA" and not str(v).startswith("SOLD OUT") for v in p.values())
        for i in range(1, 7):
            v = p.get(f"Price W{i}", "NA")
            if v == "NA" or (str(v).startswith("SOLD OUT") and not has_real):
                items.append((k, i))
    return items

def find_na_soldout_weeks(awp, keys):
    """Tìm tất cả weeks có NA hoặc SOLD OUT để auto-retry."""
    items = []
    for k in keys:
        if k not in awp: continue
        p = awp[k]
        for i in range(1, 7):
            v = p.get(f"Price W{i}", "NA")
            if v == "NA" or str(v).startswith("SOLD OUT"):
                items.append((k, i))
    return items

async def save_debug_screenshot(page, hotel_name, week_num, label):
    if not DEBUG_SCREENSHOTS:
        return
    try:
        os.makedirs(DEBUG_DIR, exist_ok=True)
        safe_name = re.sub(r'[^\w\-]', '_', hotel_name)[:30]
        path = f"{DEBUG_DIR}/{safe_name}_W{week_num}_{label}.png"
        await page.screenshot(path=path, full_page=False)
        print(f"      📸 Screenshot: {path}", flush=True)
    except:
        pass

# ============================================================
# BROWSER LAUNCHER — hỗ trợ proxy rotation
# ============================================================
_proxy_index = 0

async def launch_browser(playwright, batch_num=0):
    """Launch browser mới với fingerprint mới, optional proxy rotation."""
    global _proxy_index

    launch_args = [
        '--disable-blink-features=AutomationControlled',
        '--no-sandbox',
        '--disable-dev-shm-usage',
    ]

    proxy_config = None
    proxy_label = "direct"

    if PROXY_LIST:
        proxy_url = PROXY_LIST[_proxy_index % len(PROXY_LIST)]
        _proxy_index += 1
        proxy_config = {"server": proxy_url}
        proxy_label = proxy_url

    if HEADLESS:
        launch_args.insert(0, '--headless=new')

    kwargs = {"headless": False, "args": launch_args}
    if proxy_config:
        kwargs["proxy"] = proxy_config

    browser = await playwright.chromium.launch(**kwargs)

    mode = "headless" if HEADLESS else "visible"
    print(f"   🌐 Browser #{batch_num} launched ({mode}) | proxy: {proxy_label}", flush=True)
    return browser

# ============================================================
# CRAWL 1 NGÀY — thử 1 checkin date cụ thể
# ============================================================
async def crawl_single_day(browser, hotel_url, room_type, week_num, checkin,
                           retries=None, page_timeout=None, hotel_name=""):
    if retries is None: retries = RETRIES_PER_DAY
    if page_timeout is None: page_timeout = PAGE_TIMEOUT

    result = {"week": week_num, "price": "NA", "date": checkin.strftime('%Y-%m-%d')}

    for retry in range(retries):
        context = None
        try:
            if retry > 0:
                backoff = random.uniform(3, 6) * (retry + 1)
                await asyncio.sleep(backoff)

            context = await browser.new_context(
                viewport={"width": random.randint(1366, 1920), "height": random.randint(768, 1080)},
                user_agent=random.choice(USER_AGENTS),
                locale="en-US",
            )
            page = await context.new_page()
            await page.add_init_script(STEALTH_SCRIPT)

            url = update_url_checkin(hotel_url, checkin)

            try:
                await page.goto(url, timeout=page_timeout, wait_until="domcontentloaded")
            except Exception:
                pass

            await asyncio.sleep(random.uniform(1, 2))

            await page.evaluate("window.scrollTo({top: 300, behavior: 'smooth'})")
            await asyncio.sleep(random.uniform(0.5, 1.0))

            try:
                close_btn = page.locator(".ab-close-button")
                if await close_btn.count() > 0:
                    await close_btn.first.click(timeout=2000)
            except: pass

            try:
                await page.wait_for_selector("[data-selenium='PriceDisplay']", timeout=15000)
            except:
                try:
                    await page.wait_for_selector("div#roomGrid", timeout=8000)
                    await asyncio.sleep(3)
                except:
                    await asyncio.sleep(3)

            await page.evaluate("window.scrollTo(0, document.body.scrollHeight / 2)")
            await asyncio.sleep(0.5)
            await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
            await asyncio.sleep(random.uniform(1.0, 2.0))

            extraction = await page.evaluate(EXTRACT_PRICES_JS, room_type)

            if extraction.get('found'):
                price = extraction['price']
                partial = " (partial)" if extraction.get('partial') else ""
                result["price"] = price
                return result

            if extraction.get('soldOut'):
                st = extraction.get('soldOutType', '')
                sp = extraction.get('soldOutPrice', '')
                price = "SOLD OUT" if st == 'hotel' else f"SOLD OUT {sp}".strip()
                result["price"] = price
                return result

            # Not found — chỉ log ở retry cuối
            if retry == retries - 1:
                rooms = extraction.get('allRooms', 0)
                names = extraction.get('roomNames', [])[:3]
                result["debug_rooms"] = rooms
                result["debug_names"] = names

        except Exception as e:
            err = str(e)
            if "has been closed" in err or "Target page" in err:
                pass
        finally:
            if context:
                try: await context.close()
                except: pass

        await asyncio.sleep(random.uniform(*DELAY_RANGE))

    return result

# ============================================================
# CRAWL 1 TUẦN — thử từng ngày trong tuần cho đến khi có giá
# W1: base → base+6, W2: base+7 → base+13, ...
# ============================================================
async def crawl_week_range(browser, hotel_url, room_type, week_num, base_checkin,
                           hotel_name="", days_in_week=None, page_timeout=None):
    if days_in_week is None: days_in_week = DAYS_PER_WEEK

    week_start_offset = (week_num - 1) * 7
    week_start = base_checkin + timedelta(days=week_start_offset)
    week_end = week_start + timedelta(days=days_in_week - 1)

    print(f"      🔍 W{week_num}: trying {week_start.strftime('%m/%d')}→{week_end.strftime('%m/%d')} ...", flush=True)

    last_result = {"week": week_num, "price": "NA", "date": ""}
    has_sold_out = False
    sold_out_count = 0
    na_count = 0

    consecutive_sold = 0  # Đếm SOLD OUT liên tiếp

    for day_offset in range(days_in_week):
        checkin = base_checkin + timedelta(days=week_start_offset + day_offset)
        result = await crawl_single_day(
            browser, hotel_url, room_type, week_num, checkin,
            page_timeout=page_timeout, hotel_name=hotel_name
        )

        price = result["price"]

        # Tìm được giá → trả về ngay
        if price != "NA" and not str(price).startswith("SOLD OUT"):
            print(f"      ✅ W{week_num}: {price} | {checkin.strftime('%Y-%m-%d')} (day {day_offset+1}/{days_in_week})", flush=True)
            return result

        if str(price).startswith("SOLD OUT"):
            has_sold_out = True
            sold_out_count += 1
            consecutive_sold += 1
            last_result = result
            # Early exit: N ngày liên tiếp SOLD OUT → kết luận luôn
            if consecutive_sold >= SOLD_OUT_EARLY_EXIT:
                last_result["price"] = "SOLD OUT"
                print(f"      🚫 W{week_num}: SOLD OUT (early exit after {consecutive_sold} consecutive)", flush=True)
                return last_result
        else:
            consecutive_sold = 0  # Reset nếu gặp NA
            na_count += 1
            last_result = result

        # Delay nhỏ trước khi thử ngày tiếp
        if day_offset < days_in_week - 1:
            await asyncio.sleep(random.uniform(0.5, 1.5))

    # Hết tất cả ngày trong tuần → quyết định kết quả
    if has_sold_out:
        last_result["price"] = "SOLD OUT"
        print(f"      🚫 W{week_num}: SOLD OUT (tried {day_offset+1} days: {sold_out_count} sold, {na_count} NA)", flush=True)
    else:
        print(f"      ❌ W{week_num}: NA after trying all {days_in_week} days", flush=True)

    # Debug screenshot cho kết quả cuối
    try:
        context = await browser.new_context(
            viewport={"width": 1920, "height": 1080},
            user_agent=random.choice(USER_AGENTS), locale="en-US",
        )
        page = await context.new_page()
        await page.add_init_script(STEALTH_SCRIPT)
        # Screenshot ngày đầu tuần để debug
        url = update_url_checkin(hotel_url, week_start)
        try:
            await page.goto(url, timeout=PAGE_TIMEOUT, wait_until="domcontentloaded")
            await asyncio.sleep(3)
            label = "SOLDOUT" if has_sold_out else "NA"
            await save_debug_screenshot(page, hotel_name, week_num, label)
        except: pass
        finally:
            try: await context.close()
            except: pass
    except: pass

    return last_result

# ============================================================
# PROCESS 1 HOTEL
# ============================================================
async def process_hotel(browser, hotel_info, prev_data, base_checkin, semaphore):
    hotel_name, hotel_url, room_type = hotel_info
    key = (hotel_name, room_type)

    if key in prev_data:
        vals = [prev_data[key].get(f"Price W{i}", "NA") for i in range(1, 7)]
        all_data = all(v != "NA" for v in vals)
        all_sold = all(str(v).startswith("SOLD OUT") for v in vals)
        if all_data and not all_sold:
            return key, prev_data[key], True
        if all_sold:
            print(f"🔄 RE-CRAWL: {hotel_name} (all SOLD OUT)", flush=True)
            prev_data[key] = {f"Price W{i}": "NA" for i in range(1, 7)}

    async with semaphore:
        await asyncio.sleep(random.uniform(*HOTEL_DELAY))
        print(f"\n🏨 {hotel_name} | {room_type}", flush=True)

        prices = {}
        has_real = key in prev_data and any(
            v != "NA" and not str(v).startswith("SOLD OUT") for v in prev_data[key].values()
        )

        weeks_to_crawl = []
        for wn in range(1, 7):
            kp = f"Price W{wn}"
            cached = prev_data[key].get(kp, "NA") if key in prev_data else "NA"
            if cached != "NA" and not str(cached).startswith("SOLD OUT"):
                prices[kp] = cached
            elif str(cached).startswith("SOLD OUT") and has_real:
                prices[kp] = cached
            else:
                weeks_to_crawl.append(wn)

        if weeks_to_crawl:
            wsem = asyncio.Semaphore(WEEKS_PER_HOTEL)
            async def do_week(wn):
                async with wsem:
                    return await crawl_week_range(
                        browser, hotel_url, room_type, wn, base_checkin,
                        hotel_name=hotel_name
                    )
            results = await asyncio.gather(*[do_week(wn) for wn in weeks_to_crawl])
            for r in results:
                prices[f"Price W{r['week']}"] = r["price"]

        na_c, _ = calc_na_stats(prices)
        so_c = sum(1 for i in range(1, 7) if str(prices.get(f"Price W{i}", "")).startswith("SOLD OUT"))
        icon = "✅" if na_c == 0 and so_c == 0 else f"⚠️({na_c}NA)" if na_c else f"🚫({so_c}SO)"
        print(f"   {icon} DONE: {hotel_name}", flush=True)
        return key, prices, False

# ============================================================
# RETRY — cũng dùng crawl_week_range
# ============================================================
async def retry_batch(browser, batch_infos, batch_keys, awp, base_checkin):
    ki = {(i[0], i[2]): i for i in batch_infos}
    for rn in range(1, MAX_RETRY_ROUNDS + 1):
        nr, nc, tc = calc_batch_na_rate(awp, batch_keys)
        if nr <= TARGET_NA_RATE: break
        items = find_retry_weeks(awp, batch_keys)
        if not items: break

        to = RETRY_PAGE_TIMEOUT[min(rn-1, len(RETRY_PAGE_TIMEOUT)-1)]
        print(f"\n🔁 RETRY {rn}/{MAX_RETRY_ROUNDS} | {len(items)} cells", flush=True)
        await asyncio.sleep(random.uniform(*RETRY_COOL_DOWN))

        hna = {}
        for k, wn in items:
            hna.setdefault(k, []).append(wn)

        sem = asyncio.Semaphore(NUM_WORKERS)
        async def do_retry(k, weeks):
            async with sem:
                info = ki.get(k)
                if not info: return
                hn, hu, rt = info
                await asyncio.sleep(random.uniform(*HOTEL_DELAY))
                wsem = asyncio.Semaphore(WEEKS_PER_HOTEL)
                async def rw(wn):
                    async with wsem:
                        return await crawl_week_range(
                            browser, hu, rt, wn, base_checkin,
                            hotel_name=hn, page_timeout=to
                        )
                results = await asyncio.gather(*[rw(w) for w in weeks])
                for r in results:
                    if r["price"] != "NA":
                        awp[k][f"Price W{r['week']}"] = r["price"]
        await asyncio.gather(*[do_retry(k, w) for k, w in hna.items()])

# ============================================================
# AUTO RETRY NA & SOLD OUT — Round 2 sau khi crawl xong tất cả
# ============================================================
async def auto_retry_na_soldout(browser, all_infos, awp, base_checkin):
    """Tự động crawl lại tất cả weeks có NA hoặc SOLD OUT sau round 1."""
    ki = {(i[0], i[2]): i for i in all_infos}
    all_keys = list(awp.keys())

    items = find_na_soldout_weeks(awp, all_keys)
    if not items:
        print(f"\n✅ Không có NA/SOLD OUT nào cần retry!", flush=True)
        return 0

    hna = {}
    for k, wn in items:
        hna.setdefault(k, []).append(wn)

    na_count = sum(1 for k, wn in items if awp[k].get(f"Price W{wn}", "NA") == "NA")
    so_count = sum(1 for k, wn in items if str(awp[k].get(f"Price W{wn}", "")).startswith("SOLD OUT"))

    print(f"\n{'='*60}", flush=True)
    print(f"🔄 AUTO ROUND 2 — Re-crawl NA & SOLD OUT", flush=True)
    print(f"   📊 {len(items)} cells ({na_count} NA + {so_count} SOLD OUT) across {len(hna)} hotels", flush=True)
    print(f"{'='*60}", flush=True)

    updated = 0
    sem = asyncio.Semaphore(NUM_WORKERS)

    async def do_hotel_retry(k, weeks):
        nonlocal updated
        async with sem:
            info = ki.get(k)
            if not info: return
            hn, hu, rt = info

            await asyncio.sleep(random.uniform(*HOTEL_DELAY))
            print(f"\n   🏨 [R2] {hn} | weeks: {weeks}", flush=True)

            wsem = asyncio.Semaphore(WEEKS_PER_HOTEL)
            async def rw(wn):
                async with wsem:
                    return await crawl_week_range(
                        browser, hu, rt, wn, base_checkin,
                        hotel_name=hn, page_timeout=45000
                    )
            results = await asyncio.gather(*[rw(w) for w in weeks])
            for r in results:
                new_price = r["price"]
                old_price = awp[k].get(f"Price W{r['week']}", "NA")
                if new_price != old_price:
                    if new_price != "NA":
                        awp[k][f"Price W{r['week']}"] = new_price
                        old_label = "NA" if old_price == "NA" else "SO"
                        new_label = new_price if not str(new_price).startswith("SOLD OUT") else "SOLD OUT"
                        print(f"      🔄 W{r['week']}: {old_label} → {new_label}", flush=True)
                        updated += 1

    await asyncio.gather(*[do_hotel_retry(k, w) for k, w in hna.items()])

    print(f"\n{'─'*50}", flush=True)
    print(f"   🔄 Round 2 complete: {updated}/{len(items)} cells updated", flush=True)
    print(f"{'─'*50}", flush=True)

    return updated

# ============================================================
# MAIN
# ============================================================
async def main():
    t0 = time.time()
    if DEBUG_SCREENSHOTS:
        os.makedirs(DEBUG_DIR, exist_ok=True)

    df = read_hotels_from_csv(INPUT_FILE)
    if len(df) == 0: return

    awp, prev = {}, {}

    def load_prev(fp):
        n = 0
        try:
            dp = pd.read_csv(fp, keep_default_na=False, na_values=[])
            for _, row in dp.iterrows():
                k = (row["hotel_name"], row["room_type"])
                if k in prev:
                    for i in range(1, 7):
                        v = str(row.get(f"price_w{i}", "NA")).strip()
                        if v and v not in ("NA", "nan"):
                            prev[k][f"Price W{i}"] = v
                else:
                    p = {}
                    for i in range(1, 7):
                        v = str(row.get(f"price_w{i}", "NA")).strip()
                        p[f"Price W{i}"] = "NA" if (not v or v in ("nan", "NA")) else v
                    prev[k] = p; awp[k] = p
                n += 1
        except: pass
        return n

    if os.path.exists(TEMP_OUTPUT_FILE):
        n = load_prev(TEMP_OUTPUT_FILE)
        print(f"📂 Loaded {n} hotels from temp", flush=True)

    bc = datetime.today().replace(hour=0, minute=0, second=0, microsecond=0) + timedelta(days=1)
    infos = [(r['hotel_name'], r['hotel_url'], r['room_type']) for _, r in df.iterrows()]
    total = len(infos)

    print(f"\n📅 Week date ranges (try each day until price found):", flush=True)
    for wn in range(1, 7):
        w_start = bc + timedelta(days=(wn - 1) * 7)
        w_end = w_start + timedelta(days=DAYS_PER_WEEK - 1)
        print(f"   W{wn}: {w_start.strftime('%Y-%m-%d (%a)')} → {w_end.strftime('%Y-%m-%d (%a)')}", flush=True)

    print(f"\n{'='*60}", flush=True)
    print(f"🎭 CRAWL v8 — anti-detect + browser restart", flush=True)
    print(f"📊 {total} hotels | {NUM_WORKERS}W × {WEEKS_PER_HOTEL}wk | batch={BATCH_SIZE} | SO early-exit={SOLD_OUT_EARLY_EXIT}", flush=True)
    print(f"🔄 Browser restart: {'ON' if RESTART_BROWSER else 'OFF'} | Cooldown: {BATCH_COOLDOWN[0]}-{BATCH_COOLDOWN[1]}s", flush=True)
    print(f"🌐 Proxies: {len(PROXY_LIST) if PROXY_LIST else 'none (direct)'}", flush=True)
    print(f"🔄 Auto Round 2: {'ON' if AUTO_RETRY_NA_SOLDOUT else 'OFF'}", flush=True)
    print(f"📸 Screenshots: {DEBUG_DIR}/", flush=True)
    print(f"{'='*60}\n", flush=True)

    async with async_playwright() as p:
        browser = await launch_browser(p, batch_num=0)

        tb = (total + BATCH_SIZE - 1) // BATCH_SIZE
        ct = 0

        # ── ROUND 1: Crawl tất cả hotels ──
        for bi in range(tb):
            bs, be = bi * BATCH_SIZE, min((bi+1) * BATCH_SIZE, total)
            batch = infos[bs:be]

            # ── Restart browser giữa các batch ──
            if bi > 0 and RESTART_BROWSER:
                cooldown = random.uniform(*BATCH_COOLDOWN)
                print(f"\n🔃 Restarting browser... (cooldown {cooldown:.0f}s)", flush=True)
                try:
                    await browser.close()
                except: pass
                await asyncio.sleep(cooldown)
                browser = await launch_browser(p, batch_num=bi)

            print(f"\n{'='*60}", flush=True)
            print(f"📦 BATCH {bi+1}/{tb} | Hotels {bs+1}-{be}/{total}", flush=True)
            print(f"{'='*60}", flush=True)

            sem = asyncio.Semaphore(NUM_WORKERS)
            tasks = [process_hotel(browser, i, prev, bc, sem) for i in batch]

            bkeys = []
            for coro in asyncio.as_completed(tasks):
                try:
                    k, prices, skip = await coro
                    awp[k] = prices; bkeys.append(k)
                    if not skip: ct += 1
                    save_backup_csv(awp, TEMP_OUTPUT_FILE)
                except Exception as e:
                    print(f"❌ {e}", flush=True)

            nr, nc, tc = calc_batch_na_rate(awp, bkeys)
            print(f"\n📊 Batch {bi+1}: NA = {nr:.1%} ({nc}/{tc})", flush=True)
            if nr > TARGET_NA_RATE:
                await retry_batch(browser, batch, bkeys, awp, bc)
                save_backup_csv(awp, TEMP_OUTPUT_FILE)

            # Report
            print(f"\n{'─'*50}", flush=True)
            for k in bkeys:
                pr = awp[k]; parts = []
                for i in range(1, 7):
                    v = pr.get(f"Price W{i}", "NA")
                    parts.append("✓" if v != "NA" and not str(v).startswith("SOLD OUT") else "🚫" if str(v).startswith("SOLD OUT") else "✗")
                nac, _ = calc_na_stats(pr)
                print(f"   {'✅' if nac==0 else '⚠️'} {k[0][:35]:35s} {' '.join(parts)}", flush=True)
            print(f"{'─'*50} | ⏱️ {int((time.time()-t0)//60)}m | {ct}/{total}", flush=True)

        # ── Save file final sau Round 1 ──
        fn = f"{OUTPUT_PREFIX}{datetime.today().strftime('%Y%m%d')}.csv"
        save_backup_csv(awp, fn)
        print(f"\n📁 Round 1 saved: {fn}", flush=True)

        tc_r1 = len(awp) * 6
        na_r1 = sum(1 for p in awp.values() for i in range(1,7) if p.get(f"Price W{i}","NA") == "NA")
        so_r1 = sum(1 for p in awp.values() for i in range(1,7) if str(p.get(f"Price W{i}","")).startswith("SOLD OUT"))
        print(f"   Round 1: ✅ {tc_r1-na_r1-so_r1}/{tc_r1} | 🚫 {so_r1} SO | ❌ {na_r1} NA", flush=True)

        # ── ROUND 2: Auto retry NA & SOLD OUT (với browser mới) ──
        if AUTO_RETRY_NA_SOLDOUT and (na_r1 > 0 or so_r1 > 0):
            # Restart browser trước Round 2 để có IP/fingerprint sạch
            if RESTART_BROWSER:
                cooldown = random.uniform(*BATCH_COOLDOWN)
                print(f"\n🔃 Restarting browser for Round 2... (cooldown {cooldown:.0f}s)", flush=True)
                try:
                    await browser.close()
                except: pass
                await asyncio.sleep(cooldown)
                browser = await launch_browser(p, batch_num=999)

            r2_updated = await auto_retry_na_soldout(browser, infos, awp, bc)

            if r2_updated > 0:
                save_backup_csv(awp, fn)
                save_backup_csv(awp, TEMP_OUTPUT_FILE)
                print(f"\n📁 Final updated: {fn} ({r2_updated} cells changed)", flush=True)
            else:
                print(f"\n📁 Final unchanged (no improvements in Round 2)", flush=True)

        await browser.close()

    # ── Final stats ──
    tt = time.time() - t0; tc = len(awp) * 6
    na_t = sum(1 for p in awp.values() for i in range(1,7) if p.get(f"Price W{i}","NA") == "NA")
    so_t = sum(1 for p in awp.values() for i in range(1,7) if str(p.get(f"Price W{i}","")).startswith("SOLD OUT"))
    print(f"\n{'='*60}", flush=True)
    print(f"✅ FINAL COMPLETED | {fn}", flush=True)
    print(f"   ✅ Price: {tc-na_t-so_t}/{tc} ({(tc-na_t-so_t)/max(tc,1):.1%})", flush=True)
    print(f"   🚫 Sold:  {so_t}/{tc} | ❌ NA: {na_t}/{tc}", flush=True)
    print(f"⏱️ {int(tt//60)}m {int(tt%60)}s", flush=True)
    if DEBUG_SCREENSHOTS:
        n_shots = len([f for f in os.listdir(DEBUG_DIR) if f.endswith('.png')]) if os.path.exists(DEBUG_DIR) else 0
        if n_shots:
            print(f"\n📸 {n_shots} debug screenshots in {DEBUG_DIR}/", flush=True)
    print(f"{'='*60}", flush=True)

await main()